In [ ]:
"""
XGBoost 피처 매트릭스 조립

지금까지 따로 만든 모든 피처 소스를 하나의 (segment_key, timestamp) 단위
테이블로 합치고, 라벨(y)까지 붙여서 output/features/xgb_feature_matrix.parquet
로 저장한다.

grain(행 단위) 결정: 5분 (원본 속도 데이터 그대로)
  처음엔 Prophet 산출물과 맞춰 10분 정각(:00,:10,:20...)만 필터링하려
  했으나, 실제 speed_features의 분(minute) 값 분포를 보니 균일하지 않았다
  (0/5/15/25/35/45/55분에 각각 약 126~130만 행이 몰려있고, 10/20/30/40/50분은
  각각 약 16만 행뿐 - 원본 ITS 수집 과정에서 생긴 불균일). 10분 정각으로
  필터링하면 전체 1,058만 행 중 285만 행만 남고, 라벨(t+30min) 매칭 실패율도
  71.8%까지 치솟아 데이터 손실이 너무 컸다. 그래서 필터링 없이 원본 5분
  단위를 그대로 쓴다.

  이로 인한 영향(실측으로 검증됨, 대략치 아님):
  - Prophet(y_hat_t30 등)은 10분 단위 소스라, 5분 그리드 중 실제로 10분
    정각에 해당하는 행은 전체의 약 27%뿐이다(위에서 말한 분 분포 불균일
    때문에 "절반"이 아니다). 여기에 18개 구간 결측(20%)까지 곱해지면
    1차로는 78.5% 결측이 나온다.
  - 게다가 "10분 정각 행"은 라벨(t+30분) 매칭에서도 불리하다 - 예를 들어
    0분 행은 30분 뒤 값이 있어야 하는데, 30분대 데이터가 원래 희소해서
    매칭 실패율이 더 높다. 그래서 t+30 라벨 매칭까지 마치고 살아남은
    행만 보면 Prophet 결측률이 78.5% -> 92.2%로 더 나빠진다.
  둘 다 원본 데이터의 불균일한 분 분포에서 비롯된 실제 현상이며(재현
  검증 완료), join 코드 버그가 아니다. incident_flag(마찬가지로 10분
  단위 소스)도 같은 이유로 결측이 많다.

master grid: speed_features.parquet(가장 큰 원본, 90개 구간 전부 포함)의
  실제 관측치를 그대로 사용한다(필터링/리샘플링 없음).

결합할 피처 소스와 join key:
  1) speed_features.parquet          - (segment_key, timestamp), master
  2) is_bottleneck_slot_FINAL_named.csv - (segment_key, time_slot=hour*100+30분단위)
     ⚠ time_slot 계산 시 hour를 반드시 넓은 정수형으로 cast할 것(다음 셀 참고,
     Int8 오버플로우로 91.8% 결측 나던 버그를 실제로 겪고 수정함)
  3) network_features.parquet        - (segment_key) - 정적
  4) construction_lane_ratio_daily.parquet - (segment_id, date) - segment_id는
     segment_key에서 방향 접미사(_AB/_BA)를 뗀 것, 양방향 동일값 broadcast
  5) incident_flag_10min.parquet     - (segment_key, timestamp), sparse(True만 저장,
     10분 단위 소스라 5분 그리드의 약 27%만 매칭됨)
  6) weather_features.parquet        - (timestamp를 1시간 단위로 내림), 시 전체 공통값
  7) prophet_features.parquet        - (segment_key, timestamp), 72/90 구간만 존재,
     10분 단위 소스 + 라벨매칭과의 상호작용으로 최종 결측률 약 92%

⚠ 알려진 한계 (반드시 인지하고 사용할 것):
  1) is_bottleneck_slot은 전체 기간(639일) 데이터로 계산된 값이다.
     Train 기간만으로 다시 계산한 버전이 아니므로 엄밀히는 leakage 위험이
     있다 - bottleneck_analysis.ipynb에서 Train 기간(< 2026-06-14)으로만
     필터링해서 재계산한 뒤 이 값을 교체하는 게 이상적이다. 지금은 우선
     전체 기간 버전을 그대로 쓴다.
  2) prophet/incident 관련 컬럼은 (a) 18개 segment_key 결측(Prophet 미제공)
     + (b) 10분 단위 소스라 5분 그리드 대부분이 자연 결측(위 설명 참고)이다.
     최종 결측률이 90%를 넘는 게 정상이니 놀라지 말 것 - XGBoost가 결측을
     자체 처리하도록 둔다.
  3) 날씨는 대전 단일 관측소 기준이라 90개 구간 모두 동일한 값을 받는다
     (공간 해상도 없음).
  4) split(Train/Val/Test)은 prophet_features의 split 컬럼과 동일한 날짜
     기준(Train < 2026-06-14, Val 2026-06-14~06-27, Test >= 2026-06-28)을
     모든 90개 구간에 독립적으로 다시 적용한 것이다(조인된 split 컬럼은
     72개 구간에만 있어 그대로 못 씀).

출력: output/features/xgb_feature_matrix.parquet
"""

from pathlib import Path

import polars as pl

FEATURES_DIR = Path("./output/features")
EDA_DIR = Path("./output/eda")

SPEED_PATH = FEATURES_DIR / "speed_features.parquet"
BOTTLENECK_PATH = EDA_DIR / "is_bottleneck_slot_FINAL_named.csv"
NETWORK_PATH = FEATURES_DIR / "network_features.parquet"
LANE_RATIO_PATH = FEATURES_DIR / "construction_lane_ratio_daily.parquet"
INCIDENT_PATH = FEATURES_DIR / "incident_flag_10min.parquet"
WEATHER_PATH = FEATURES_DIR / "weather_features.parquet"
PROPHET_PATH = FEATURES_DIR / "prophet_features.parquet"

OUTPUT_PATH = FEATURES_DIR / "xgb_feature_matrix.parquet"

# split 경계 - prophet_features.ipynb에서 검증된 것과 동일하게 고정
TRAIN_END = "2026-06-14"   # 이 날짜 미만 = Train
VAL_END = "2026-06-28"     # 이 날짜 미만 = Val, 이상 = Test

TS_UNIT = "us"  # speed_features의 timestamp 정밀도(us)에 나머지 소스를 맞춤


In [ ]:
# ==================================================================
# 1. master grid: 원본 5분 단위 그대로 로드 (필터링 없음)
# ==================================================================

speed = pl.read_parquet(SPEED_PATH).with_columns(
    pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
)
print(f"master shape: {speed.shape}")

df = speed
print(f"segment_key 수: {df['segment_key'].n_unique()}")
print(f"기간: {df['timestamp'].min()} ~ {df['timestamp'].max()}")


In [ ]:
# ==================================================================
# 2. 시간 파생 피처 + 조인용 보조 키(time_slot, date, weather_hour, segment_id)
# ==================================================================
# ⚠ time_slot 계산 시 주의: pl.col("timestamp").dt.hour()는 Int8을 반환하는데,
# 여기에 *100을 곱하면 hour>=2부터 Int8 범위(-128~127)를 넘어 오버플로우가
# 난다(hour=2 -> 200). 반드시 더 넓은 정수형으로 cast한 뒤 곱해야 한다.
# (이 오버플로우 버그로 인해 is_bottleneck_slot 조인이 91.8% 결측나는 문제를
# 실제로 겪었음 - hour<2인 행만 정상 매칭되고 나머지는 다 실패했었음)

df = df.with_columns(
    [
        pl.col("timestamp").dt.hour().alias("hour"),
        (pl.col("timestamp").dt.weekday() - 1).alias("dow"),  # polars: 1=월 -> 0=월로 맞춤
        pl.col("timestamp").dt.date().alias("date"),
        pl.col("timestamp").dt.truncate("1h").cast(pl.Datetime(TS_UNIT)).alias("weather_hour"),
        # segment_key("SEG_01_201_202_AB") -> segment_id("SEG_01_201_202") : 마지막 "_AB"/"_BA" 제거
        pl.col("segment_key").str.slice(0, pl.col("segment_key").str.len_chars() - 3).alias("segment_id"),
    ]
).with_columns(
    [
        (pl.col("dow") >= 5).alias("is_weekend"),
        (
            (pl.col("hour").cast(pl.Int32) * 100)
            + (pl.col("timestamp").dt.minute() // 30) * 30
        ).alias("time_slot"),
    ]
)

print(df.select(["segment_key", "segment_id", "timestamp", "hour", "dow", "is_weekend", "time_slot", "date"]).head())

# time_slot 검증: bottleneck lookup에 있는 48개 값 밖으로 나가는 값이 없어야 정상
bottleneck_check = pl.read_csv(BOTTLENECK_PATH).select("time_slot").unique()
bad_slots = df.filter(~pl.col("time_slot").is_in(bottleneck_check["time_slot"])).height
print(f"time_slot 검증(0이어야 정상): lookup에 없는 값을 가진 행 수 = {bad_slots}")
assert bad_slots == 0, "time_slot 계산에 문제가 있습니다 - 오버플로우 등 확인 필요"


In [ ]:
# ==================================================================
# 3. is_bottleneck_slot 조인
# ==================================================================
# ⚠ 전체 기간으로 계산된 값 (leakage 위험, 개요 셀 참고)

bottleneck = pl.read_csv(BOTTLENECK_PATH).select(["segment_key", "time_slot", "is_bottleneck_slot"])

before = df.height
df = df.join(bottleneck, on=["segment_key", "time_slot"], how="left")
print(f"조인 후 shape: {df.shape} (조인 전 {before}행 유지되어야 정상)")
print(f"is_bottleneck_slot 결측: {df['is_bottleneck_slot'].null_count()}")

In [ ]:
# ==================================================================
# 4. network_features 조인 (정적, segment_key 기준)
# ==================================================================

network = pl.read_parquet(NETWORK_PATH)
df = df.join(network, on="segment_key", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"결측 확인:\n{df.select(['betweenness_pre', 'betweenness_during', 'road_rank', 'lanes']).null_count()}")

In [ ]:
# ==================================================================
# 5. construction_lane_ratio_daily 조인 (segment_id + date)
# ==================================================================

lane_ratio = pl.read_parquet(LANE_RATIO_PATH).with_columns(
    pl.col("date").dt.date().alias("date")
)

df = df.join(lane_ratio, on=["segment_id", "date"], how="left").with_columns(
    pl.col("lane_remain_ratio").fill_null(1.0)  # 매칭 안 되면 자유흐름으로 간주
)
print(f"조인 후 shape: {df.shape}")
print(f"lane_remain_ratio 결측(채움 후 0이어야 정상): {df['lane_remain_ratio'].null_count()}")

In [ ]:
# ==================================================================
# 6. incident_flag_10min 조인 (segment_key + timestamp, sparse)
# ==================================================================

incident = pl.read_parquet(INCIDENT_PATH).with_columns(
    pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
)

df = df.join(incident, on=["segment_key", "timestamp"], how="left").with_columns(
    [
        pl.col("incident_flag").fill_null(False),
        pl.col("incident_count").fill_null(0),
    ]
)
print(f"조인 후 shape: {df.shape}")
print(f"incident_flag=True 행 수: {df['incident_flag'].sum()}")

In [ ]:
# ==================================================================
# 7. weather_features 조인 (1시간 단위, 시 전체 공통값)
# ==================================================================

weather = pl.read_parquet(WEATHER_PATH).with_columns(
    pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
).rename({"timestamp": "weather_hour"})

df = df.join(weather, on="weather_hour", how="left")
print(f"조인 후 shape: {df.shape}")
print(f"결측 확인:\n{df.select(['precipitation_mm', 'is_weather_alert', 'is_freezing']).null_count()}")

In [ ]:
# ==================================================================
# 8. prophet_features 조인 (segment_key + timestamp, 72/90 구간만 존재)
# ==================================================================
# split 컬럼은 72개 구간에만 존재해서 그대로 못 쓴다(prophet_split으로만
# 참고 보관) - 실제 split은 다음 셀에서 전체 90개 구간에 독립적으로 부여한다.

if PROPHET_PATH.exists():
    prophet = pl.read_parquet(PROPHET_PATH).with_columns(
        pl.col("timestamp").cast(pl.Datetime(TS_UNIT))
    ).rename({"split": "prophet_split"})

    df = df.join(prophet, on=["segment_key", "timestamp"], how="left")
    print(f"조인 후 shape: {df.shape}")
    print(f"y_hat_t30 결측 비율: {df['y_hat_t30'].null_count() / df.height:.1%} (18개 미커버 구간 + 커버리지 밖 날짜)")
else:
    print(f"⚠ {PROPHET_PATH} 없음 - prophet_features.ipynb를 먼저 실행하세요. 이번엔 prophet 컬럼 없이 진행합니다.")

In [ ]:
# ==================================================================
# 9. 라벨(y) 생성: V_segment(t+30min) 기준 3-class
# ==================================================================
# t 시점 기준 정확히 30분 뒤 실측치를 찾아 라벨링한다. 30분 뒤 관측치가
# 없는 행(결측 구간, 데이터 끝자락)은 라벨을 만들 수 없으므로 이후
# drop한다 - 클래스 분포 확인 때 확인된 대로 약 22% 정도 유실 예상.

target = speed.select(
    [
        pl.col("segment_key"),
        pl.col("timestamp").alias("target_ts"),
        pl.col("V_segment").alias("target_speed"),
    ]
)

df = df.with_columns(
    (pl.col("timestamp") + pl.duration(minutes=30)).alias("target_ts")
).join(target, on=["segment_key", "target_ts"], how="left")

n_before = df.height
df = df.filter(pl.col("target_speed").is_not_null())
print(f"라벨 매칭 실패로 제거된 행: {n_before - df.height} ({(n_before - df.height) / n_before:.1%})")

df = df.with_columns(
    pl.when(pl.col("target_speed") >= 20)
    .then(0)
    .when(pl.col("target_speed") >= 15)
    .then(1)
    .otherwise(2)
    .alias("label")
)

print(df["label"].value_counts().sort("label"))

In [ ]:
# ==================================================================
# 10. split(Train/Val/Test) 부여 - 90개 구간 전체에 동일 날짜 기준 적용
# ==================================================================

df = df.with_columns(
    pl.when(pl.col("timestamp") < pl.lit(TRAIN_END).str.to_datetime())
    .then(pl.lit("Train"))
    .when(pl.col("timestamp") < pl.lit(VAL_END).str.to_datetime())
    .then(pl.lit("Val"))
    .otherwise(pl.lit("Test"))
    .alias("split")
)

print(df.group_by("split").agg(pl.len()).sort("split"))
print()
print("split별 라벨 분포:")
print(df.group_by(["split", "label"]).agg(pl.len()).sort(["split", "label"]))

In [ ]:
# ==================================================================
# 11. 최종 컬럼 정리 + 저장
# ==================================================================

final_cols = [
    "segment_key", "timestamp", "split",
    # 실측 속도
    "V_segment", "speed_last_10min", "speed_ma_30min", "speed_ma_1h", "speed_change_rate",
    # 시간
    "hour", "dow", "is_weekend", "is_bottleneck_slot",
    # 네트워크
    "betweenness_pre", "betweenness_during", "road_rank", "lanes",
    # 공사
    "lane_remain_ratio",
    # 돌발
    "incident_flag", "incident_count",
    # 기상
    "precipitation_mm", "is_weather_alert", "is_freezing",
    # 라벨
    "target_speed", "label",
]
if "y_hat_t30" in df.columns:
    final_cols += ["y_hat_t30", "y_hat_lower_t30", "y_hat_upper_t30", "prophet_split"]

final_df = df.select(final_cols)
final_df.write_parquet(OUTPUT_PATH)

print(f"저장 완료: {OUTPUT_PATH.resolve()}")
print(f"최종 shape: {final_df.shape}")
print(final_df.schema)

In [ ]:
# ==================================================================
# 12. 최종 검증: 컬럼별 결측률 + 요약
# ==================================================================

null_summary = final_df.null_count().transpose(include_header=True, header_name="column", column_names=["null_count"])
null_summary = null_summary.with_columns((pl.col("null_count") / final_df.height * 100).round(2).alias("null_pct"))
print(null_summary.sort("null_pct", descending=True))

print()
print("=== 요약 ===")
print(f"총 행 수: {final_df.height:,}")
print(f"segment_key 수: {final_df['segment_key'].n_unique()}")
print(f"기간: {final_df['timestamp'].min()} ~ {final_df['timestamp'].max()}")
print()
print("※ 알려진 한계(개요 셀 참고): is_bottleneck_slot leakage 가능성 / "
      "prophet 18개 구간 결측 / 날씨 공간 해상도 없음 - 실제 모델 학습 전 재확인 권장")